# Повышение разрешения изображений автокодировщиком (Fashion-MNIST)
Задача: из монохромного изображения **7×7** получить изображение **28×28**.

- Модель: свёрточный автокодировщик (encoder → decoder).
- Данные: Fashion-MNIST. Вход 7×7 получаем уменьшением оригинала 28×28 в 4 раза.
- Сравнение: бикубическая интерполяция 7×7 → 28×28.
- Метрики: MSE, PSNR, SSIM на всей тестовой выборке (10 000 изображений).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

tf.keras.utils.set_random_seed(42)

(x_train, _), (x_test, _) = tf.keras.datasets.fashion_mnist.load_data()

x_train = np.expand_dims(x_train.astype("float32") / 255, -1)
x_test = np.expand_dims(x_test.astype("float32") / 255, -1)

x_train_lr = tf.image.resize(x_train, (7, 7), method="area").numpy()
x_test_lr = tf.image.resize(x_test, (7, 7), method="area").numpy()

print(x_train.shape, x_train_lr.shape, x_test.shape, x_test_lr.shape)

In [ ]:
encoder = tf.keras.Sequential([
    tf.keras.Input(shape=(7, 7, 1)),
    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
], name="encoder")

decoder = tf.keras.Sequential([
    tf.keras.Input(shape=(7, 7, 128)),
    tf.keras.layers.Conv2DTranspose(64, 3, strides=2, padding="same", activation="relu"),
    tf.keras.layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu"),
    tf.keras.layers.Conv2D(1, 3, padding="same", activation="sigmoid"),
], name="decoder")

autoencoder = tf.keras.Sequential([encoder, decoder], name="autoencoder")
autoencoder.compile(optimizer="adam", loss="mse")
autoencoder.summary()

history = autoencoder.fit(x_train_lr, x_train, batch_size=128, epochs=15, validation_split=0.1)

In [ ]:
sr_ae = autoencoder.predict(x_test_lr, batch_size=256)
sr_bicubic = tf.clip_by_value(tf.image.resize(x_test_lr, (28, 28), method="bicubic"), 0.0, 1.0).numpy()

def metrics(pred, true):
    return {
        "MSE": float(np.mean((pred - true) ** 2)),
        "PSNR, дБ": float(tf.reduce_mean(tf.image.psnr(pred, true, max_val=1.0))),
        "SSIM": float(tf.reduce_mean(tf.image.ssim(pred, true, max_val=1.0))),
    }

results = {"Бикубическая": metrics(sr_bicubic, x_test), "Автокодировщик": metrics(sr_ae, x_test)}

print(f"{'Метод':<16}" + "".join(f"{k:>12}" for k in results["Бикубическая"]))
for name, m in results.items():
    print(f"{name:<16}" + "".join(f"{v:>12.4f}" for v in m.values()))

assert results["Автокодировщик"]["MSE"] < results["Бикубическая"]["MSE"]
assert results["Автокодировщик"]["SSIM"] > results["Бикубическая"]["SSIM"]
print("Автокодировщик точнее бикубической интерполяции")

In [ ]:
titles = ["Вход 7×7", "Бикубическая", "Автокодировщик", "Оригинал 28×28"]
fig, axes = plt.subplots(10, 4, figsize=(8, 20))

for col, name in enumerate(titles):
    axes[0, col].set_title(name, fontsize=11)

for i in range(10):
    for col, img in enumerate([x_test_lr[i], sr_bicubic[i], sr_ae[i], x_test[i]]):
        axes[i, col].imshow(img[:, :, 0], cmap="gray", vmin=0, vmax=1)
        axes[i, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
plt.plot(history.history["loss"], label="обучение")
plt.plot(history.history["val_loss"], label="валидация")
plt.axhline(results["Бикубическая"]["MSE"], color="gray", linestyle="--", label="бикубическая (тест)")
plt.xlabel("эпоха")
plt.ylabel("MSE")
plt.legend()
plt.show()